# Lab 05: Star Schema Blueprint (DBML), DuckDB SQL Console & Conformed Dimensions
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Date & Format:** Lab #5 (EN: Fri 27.11.2026 / FR: Mon 30.11.2026) · 3 Hours (180 min)  
**Companion Lecture:** Follows directly from **Lecture 04** (*Data Modeling, Kimball Star Schema & SQL*) and prepares your squad for **Tutorial #2**!

---

## 🕵️ Today's Story: Factory Detective in Native DuckDB SQL
No heavy Python or Pandas code today! You will work directly inside **DuckDB's visual SQL Console**—seeing your tables, column histograms, and query results side-by-side—to test management's claim about **Case 0 (`EuroPack Beverage Bottling — Plant #1, Gdańsk`)**:
> *"We think `LINE_A` is our biggest breakdown problem, and our night shifts seem fine. Check the real numbers—and build the foundation of our Gold Data Warehouse!"*

**Your 4-Step Flow Today:**
1. **Visual Blueprint (`dbdiagram.io` · 25 min):** Repair 4 bugs in EuroPack's `schema.dbml` and see how 15-minute Silver telemetry connects to Shift Production in Gold.
2. **Stage Silver Data & Open DuckDB SQL Console (15 min):** Load all 6 clean Silver Parquets into `europack_lab05.duckdb`, auto-load your **native SQL notebook**, and explore tables visually in the DuckDB sidebar.
3. **SQL Detective Missions, JOINs & 4 Dimensions (Inside DuckDB Console · 115 min):** Start from **Mission 0** (simple `SELECT`/`WHERE`/`LIMIT`), work through `GROUP BY`, `%` shares, JOINs, CTEs, the Live Fan-Out Poll (`31.9×`!), and build `Dim_Line`, `Dim_Shift`, `Dim_Format`, and `Dim_Date` natively in SQL!
4. **Final Checkpoint & Parquet Export (10 min):** Run one Jupyter cell to verify all 4 Dimensions (`✅`), export Gold Parquets, and close `europack_lab05.duckdb` cleanly for **Lab 06**.

---

## 0. Setup & Helper Functions *(Run Once)*

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case0_europack_bottling"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )
import json
import re
import uuid
import duckdb
import pandas as pd

SILVER_DIR = resolve_stage_dir("silver")
GOLD_OUT = SANDBOX_DATA / "gold"
GOLD_OUT.mkdir(parents=True, exist_ok=True)
DB_PATH = GOLD_OUT / "europack_lab05.duckdb"
if DB_PATH.exists():
    DB_PATH.unlink()

con = duckdb.connect(str(DB_PATH))


def check_dbml(spec: str) -> None:
    """Verify all 4 modeling bugs in schema.dbml are fixed (including [not null, ref: > ...])."""
    issues = []
    required_tables = [
        "Dim_Date", "Dim_Shift", "Dim_Line", "Dim_Format",
        "Fact_Shift_Production", "Fact_Stoppage_Events", "Fact_Telemetry_15min",
    ]
    for tbl in required_tables:
        if not re.search(rf"Table\s+{tbl}\b", spec):
            issues.append(f"Missing table: {tbl}")
    if "oee_pct" in spec.lower():
        issues.append("Bug #4 still present: non-additive ratio 'oee_pct' should NOT be stored in Fact_Shift_Production!")
    if re.search(r"Table\s+Fact_Shift_Production\s*\{[^}]*line_name", spec, re.S):
        issues.append("Bug #4 still present: descriptive text 'line_name' belongs in Dim_Line, not Fact_Shift_Production!")
    if re.search(r"ref:\s*<\s*Dim_", spec):
        issues.append("Bug #3 still present: wrong Ref direction ('<'). Many facts point to ONE dimension using '[not null, ref: > Dim_...]'!")
    fk_not_null = len(re.findall(r"\[[^\]]*not\s+null[^\]]*ref:\s*>\s*Dim_|\[[^\]]*ref:\s*>\s*Dim_[^\]]*not\s+null", spec, re.I))
    ref_count = len(re.findall(r"ref:\s*>\s*Dim_", spec))
    if ref_count < 11:
        issues.append(f"Expected 11 foreign-key links '[not null, ref: > Dim_...]' (4 on Fact_Shift_Production + 3 on Fact_Stoppage_Events + 4 on Fact_Telemetry_15min), found {ref_count}.")
    elif fk_not_null < 11:
        issues.append(f"dbdiagram.io requires 'not null' on mandatory '>' foreign keys: found {fk_not_null}/11 with 'not null' (use '[not null, ref: > Dim_...]').")
    if issues:
        print("❌ DBML still has modeling issue(s):")
        for msg in issues:
            print("   •", msg)
    else:
        (GOLD_OUT / "schema.dbml").write_text(spec.strip() + "\n", encoding="utf-8")
        print("✅ DBML Star Schema PASSED! All 7 tables (4 Dims + 3 Facts), 11 NOT NULL FK links, and additivity rules verified.")
        print(f"   Saved -> {GOLD_OUT / 'schema.dbml'}")


def seed_duckdb_ui_notebook(
    title: str,
    sql_cells: list[dict | str],
    db_name: str = "europack_lab05",
    force_refresh: bool = False,
    make_active: bool = True,
) -> bool:
    """Load a pre-built native SQL notebook into DuckDB's local SQL Console (idempotent!)."""
    try:
        loc = con.execute(
            "SELECT database_name, schema_name FROM duckdb_tables() WHERE table_name = 'notebook_versions' LIMIT 1"
        ).fetchone()
        if not loc:
            return False
        ui_cat, ui_sch = loc
        existing = con.execute(
            f"SELECT notebook_id, version FROM {ui_cat}.{ui_sch}.notebook_versions WHERE title = ? AND expires IS NULL ORDER BY version DESC LIMIT 1",
            [title],
        ).fetchone()

        def _build_payload(ver: int) -> str:
            cells_out = []
            for i, item in enumerate(sql_cells, start=1):
                q = item["query"] if isinstance(item, dict) else str(item)
                cells_out.append(
                    {
                        "query": q,
                        "useDatabase": db_name,
                        "cellId": i,
                        "isActive": (i == 1),
                        "runMode": "default",
                    }
                )
            return json.dumps(
                {
                    "cells": cells_out,
                    "currentDatabase": db_name,
                    "viewMode": {"mode": "default"},
                    "version": ver,
                    "notebookSerializationFormat": 3,
                }
            )

        def _activate(nid: str) -> None:
            if make_active:
                con.execute(f"DELETE FROM {ui_cat}.{ui_sch}.current_notebook_id")
                con.execute(f"INSERT INTO {ui_cat}.{ui_sch}.current_notebook_id VALUES (?)", [nid])

        if existing and not force_refresh:
            nb_id, ver = existing
            if ver == 1:
                con.execute(
                    f"UPDATE {ui_cat}.{ui_sch}.notebook_versions SET json = ? WHERE notebook_id = ? AND version = 1",
                    [_build_payload(1), nb_id],
                )
                _activate(nb_id)
                print(f"🔄 Refreshed unedited DuckDB Console notebook '{title}' ({len(sql_cells)} cells)!")
                return True
            _activate(nb_id)
            print(f"📖 DuckDB Console notebook '{title}' already exists (preserving your edits — set FORCE_REFRESH_NOTEBOOK = True to reset!).")
            return True

        if existing and force_refresh:
            nb_id, old_ver = existing
            new_ver = int(old_ver) + 1
            con.execute(
                f"UPDATE {ui_cat}.{ui_sch}.notebook_versions SET expires = now() WHERE notebook_id = ? AND expires IS NULL",
                [nb_id],
            )
            con.execute(
                f"INSERT INTO {ui_cat}.{ui_sch}.notebook_versions (notebook_id, version, title, json, created, expires) VALUES (?, ?, ?, ?, now(), NULL)",
                [nb_id, new_ver, title, _build_payload(new_ver)],
            )
            _activate(nb_id)
            print(f"🔄 Reset DuckDB Console notebook '{title}' -> version {new_ver} ({len(sql_cells)} cells)!")
            return True

        nb_id = str(uuid.uuid4())
        nb_name = f"notebook_{nb_id[:12]}"
        con.execute(f"INSERT INTO {ui_cat}.{ui_sch}.notebooks (id, name, created) VALUES (?, ?, now())", [nb_id, nb_name])
        con.execute(
            f"INSERT INTO {ui_cat}.{ui_sch}.notebook_versions (notebook_id, version, title, json, created, expires) VALUES (?, 1, ?, ?, now(), NULL)",
            [nb_id, title, _build_payload(1)],
        )
        _activate(nb_id)
        print(f"✅ Pre-loaded DuckDB Console notebook '{title}' ({len(sql_cells)} cells)!")
        return True
    except Exception as exc:
        print(f"ℹ️ Could not auto-seed DuckDB Console notebook ({exc}). Use the backup .sql worksheet file instead!")
        return False


def verify_and_export_dimensions() -> bool:
    """Verify all 4 Conformed Dimensions built inside DuckDB Console and export Gold Parquet files."""
    existing = set(
        con.execute("SELECT table_name FROM information_schema.tables WHERE table_type='BASE TABLE'").df()["table_name"]
    )
    required = ("Dim_Date", "Dim_Shift", "Dim_Line", "Dim_Format")
    missing = [t for t in required if t not in existing]
    if missing:
        print(f"💡 Still waiting for Dimension table(s) to be created in DuckDB Console: {missing}")
        print("   Complete Gold Build 1–3 in the DuckDB SQL Console, then re-run this cell!")
        return False

    counts = {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in required}
    wk = con.execute("SELECT COUNT(*) FROM Dim_Date WHERE is_weekend = TRUE").fetchone()[0]
    s_min = con.execute("SELECT MIN(planned_duration_min) FROM Dim_Shift").fetchone()[0]
    expected = {"Dim_Date": 60, "Dim_Shift": 3, "Dim_Line": 4, "Dim_Format": 3}

    if counts != expected or wk != 18 or s_min != 480:
        print(f"❌ Dimension check mismatch -> counts={counts} (expected {expected}), weekend_days={wk} (expected 18), shift_min={s_min} (expected 480).")
        return False

    exported = []
    for tbl in required:
        out_p = GOLD_OUT / f"{tbl.lower()}.parquet"
        con.execute(f"COPY {tbl} TO '{out_p}' (FORMAT PARQUET, COMPRESSION SNAPPY)")
        exported.append(out_p.name)

    print(f"✅ ALL 4 CONFORMED DIMENSIONS VERIFIED & EXPORTED TO {GOLD_OUT}:")
    print("   Parquet files:", ", ".join(exported))
    print(con.execute("""
        SELECT 'Dim_Date' AS dimension_table, COUNT(*) AS row_count FROM Dim_Date
        UNION ALL SELECT 'Dim_Shift',  COUNT(*) FROM Dim_Shift
        UNION ALL SELECT 'Dim_Line',   COUNT(*) FROM Dim_Line
        UNION ALL SELECT 'Dim_Format', COUNT(*) FROM Dim_Format
    """).df().to_string(index=False))
    return True


print("Silver Source :", SILVER_DIR)
print("DuckDB Target :", DB_PATH)
print("DuckDB Version:", duckdb.__version__)


## 1. Visual Blueprint: Repair the Star Schema (`dbdiagram.io` · 25 min)
### Why write DBML before SQL? (And why 15-min Telemetry becomes Shift Production!)
* **DBML = The CAD Blueprint of Your Warehouse:** Just as you never machine a part without a CAD drawing, `schema.dbml` lets your squad catch missing tables, broken join arrows, or invalid metrics in **2 minutes** before writing SQL—and each `Table` block translates 1-to-1 into a physical DuckDB table!
* **Kimball Principle — Analytical Convenience Beats Avoiding Duplication:** In Gold (Lab 06), we intentionally keep **BOTH** an atomic Fact table (**`Fact_Telemetry_15min`**, `23,000` rows — preserving every cleaned 15-min temperature/pressure reading!) **AND** a pre-aggregated Fact table (**`Fact_Shift_Production`**, `1,010` rows — pro-rated `480 min` shift OEE & weighted process summaries via `telemetry_readings`), plus **`Fact_Stoppage_Events`** (`289` rows). That way **dashboards query Gold exclusively—never Silver!**
* **Why `Dim_Date` (`YYYYMMDD` integer) & `NOT NULL` Foreign Keys?** A dedicated `Dim_Date` gives gap-free trends, ready-made `is_weekend`/`quarter` slicers, and one calendar filtering *both* Fact tables. Using a `YYYYMMDD` integer key (`20261001`) is human-readable, naturally sortable, fast to join, avoids timezone/format bugs, and supports *'Unknown/TBD'* sentinels without `NULL`s. Every Fact FK uses **`[not null, ref: > Dim_...]`**!

### Pair Activity: Fix the 4 Planted Bugs on [dbdiagram.io/d](https://dbdiagram.io/d)
1. Add missing **`Dim_Date`** (`date_key [pk]`, `calendar_date`, `day_of_week`, `day_of_month`, `month_name`, `quarter`, `year`, `is_weekend`).
2. Add missing foreign keys (`date_key`, `shift_key`, `format_key`) to **`Fact_Shift_Production`** using `[not null, ref: > Dim_...]`.
3. Fix the reversed arrow (`<` → `>`) on `Fact_Stoppage_Events.line_key` and add `date_key` & `shift_key`.
4. Remove Kimball violations (`line_name` text and non-additive `oee_pct`) from **`Fact_Shift_Production`**.

<details><summary>💡 Stuck on DBML syntax? Click for an example</summary>

```dbml
Table Dim_Date {
  date_key integer [pk]
  calendar_date date
  day_of_week varchar
  day_of_month integer
  month_name varchar
  quarter integer
  year integer
  is_weekend boolean
}
// Foreign keys inside Fact_Shift_Production (notice 'not null'!):
  date_key integer [not null, ref: > Dim_Date.date_key]
  shift_key integer [not null, ref: > Dim_Shift.shift_key]
  line_key integer [not null, ref: > Dim_Line.line_key]
  format_key integer [not null, ref: > Dim_Format.format_key]
```
</details>

In [ ]:
# Paste this BROKEN DBML into https://dbdiagram.io/d and fix all 4 bugs visually with your partner,
# then paste your clean DBML back into dbml_spec below and run this cell!
#
# THE 4 PLANTED BUGS TO FIX:
#   Bug #1: Dim_Date is missing! Add Table Dim_Date (date_key [pk], calendar_date, day_of_week,
#           day_of_month, month_name, quarter, year, is_weekend).
#   Bug #2: Fact_Shift_Production is missing foreign keys: date_key, shift_key, format_key.
#           (Remember: dbdiagram.io requires '[not null, ref: > Dim_...]' on mandatory '>' FKs!)
#   Bug #3: Fact_Stoppage_Events has a WRONG arrow ('<' instead of '>') on line_key,
#           and is missing date_key and shift_key foreign keys.
#   Bug #4: Fact_Shift_Production violates Kimball rules by storing descriptive text ('line_name')
#           AND a non-additive ratio ('oee_pct')! Remove both from the Fact table.

dbml_spec = '''// EuroPack Beverage Bottling — BROKEN STARTER (Fix the 4 Bugs on dbdiagram.io!)
Table Dim_Shift {
  shift_key integer [pk]
  shift_code varchar [unique]
  shift_name varchar
  start_time varchar
  end_time varchar
  planned_duration_min integer
}

Table Dim_Line {
  line_key integer [pk]
  line_code varchar [unique]
  line_name varchar
  filler_type varchar
  nominal_cycle_time_sec double
  hourly_operating_cost_eur double
}

Table Dim_Format {
  format_key integer [pk]
  format_code varchar [unique]
  format_name varchar
  packaging_material varchar
  target_hourly_output integer
  unit_reject_cost_eur double
}

Table Fact_Shift_Production {
  fact_id integer [pk]
  line_key integer [not null, ref: > Dim_Line.line_key]
  // TODO (Bug #2): Add date_key, shift_key, format_key with [not null, ref: > Dim_...]
  line_name varchar          // <-- Bug #4: Descriptive text belongs in Dim_Line!
  oee_pct double             // <-- Bug #4: Non-additive ratio! Compute in SQL views, never store in Fact!
  planned_time_min double
  unplanned_downtime_min double
  operating_time_min double
  bottled_units bigint
  good_units bigint
  rejected_units bigint
  telemetry_readings integer
  avg_pressure_bar double
  avg_pasteurizer_temp_C double
  avg_delta_temp_C double
  avg_valve_cycles_k double
  max_valve_cycles_k double
  unplanned_downtime_cost_eur double
  reject_material_cost_eur double
}

Table Fact_Stoppage_Events {
  log_key integer [pk]
  log_id varchar [unique]
  line_key integer [not null, ref: < Dim_Line.line_key]  // <-- Bug #3: Wrong arrow direction!
  // TODO (Bug #3): Add date_key and shift_key with [not null, ref: > Dim_...]
  event_timestamp timestamp
  stoppage_type varchar
  root_cause varchar
  technician_team varchar
  downtime_min double
  downtime_cost_eur double
  pre_stop_max_valve_cycles_k double
  pre_stop_max_delta_temp_C double
  pre_stop_max_pressure_bar double
  is_valve_wear_zone boolean
}

// Atomic 15-min Telemetry Fact (already properly linked — use as a syntax example!)
// Kimball lesson: Keeping BOTH Fact_Telemetry_15min (atomic) AND Fact_Shift_Production
// (aggregated) gives full analytical power in Gold without ever reaching back to Silver!
Table Fact_Telemetry_15min {
  telemetry_key integer [pk]
  date_key integer [not null, ref: > Dim_Date.date_key]
  shift_key integer [not null, ref: > Dim_Shift.shift_key]
  line_key integer [not null, ref: > Dim_Line.line_key]
  format_key integer [not null, ref: > Dim_Format.format_key]
  event_timestamp timestamp
  ambient_temp_C double
  pasteurizer_temp_C double
  delta_temp_C double
  conveyor_speed_bpm bigint
  filler_pressure_bar double
  valve_cycles_k bigint
  bottled_units bigint
  rejected_units bigint
  is_imputed_flag boolean
}
'''

check_dbml(dbml_spec)


## 2. Load Silver Data & Open the DuckDB SQL Console (15 min)
Run the cell below once. It loads all 6 clean Silver Parquet files into `europack_lab05.duckdb`, opens the **DuckDB SQL Console (`http://localhost:4213`)**, and pre-loads **two notebooks**: your clean main workbook **`Lab 05 — EuroPack SQL Missions`** (active) plus a separate **`Lab 05 — 💡 Hints & Solutions`** notebook in the left sidebar (only open if stuck!)!

### First 5 Minutes Inside DuckDB Console (No SQL Required Yet!)
In the **left Database Explorer panel** (`europack_lab05 → main`), click through your 6 staging tables:
* **`stg_lines`** (`4` rows) — Bottling lines, cycle times & hourly downtime cost (`€/h`).
* **`stg_shifts`** (`3` rows) — `S1`, `S2`, `S3` (`8` hours = `480` min).
* **`stg_formats`** (`3` rows) — `PET_05`, `PET_15`, `GLASS_075` & unit reject cost.
* **`stg_stoppages`** (`289` rows) — Maintenance breakdown logs (*check the automatic column histograms!*).
* **`stg_telemetry`** (`23,000` rows, 15-min grain) vs. **`stg_shift_integrated`** (`1,010` rows, shift-format grain).

In [ ]:
# 1. Materialize all 6 clean Silver Parquet files as portable DuckDB tables (stg_*)
for tbl, fname in [
    ("stg_telemetry", "silver_telemetry.parquet"),
    ("stg_stoppages", "silver_stoppages.parquet"),
    ("stg_lines", "silver_lines.parquet"),
    ("stg_shifts", "silver_shifts.parquet"),
    ("stg_formats", "silver_bottle_formats.parquet"),
    ("stg_shift_integrated", "silver_shift_integrated.parquet"),
]:
    p = SILVER_DIR / fname
    if p.exists():
        con.execute(f"CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM read_parquet('{p}')")

print("✅ Loaded 6 staging tables into europack_lab05.duckdb:")
print(con.execute("SHOW TABLES").df().to_string(index=False))

# 2. Launch DuckDB's Native SQL Console & Pre-Load TWO Notebooks:
#      • Active notebook  : 'Lab 05 — EuroPack SQL Missions' (clean exercises — zero spoilers!)
#      • Sidebar notebook : 'Lab 05 — 💡 Hints & Solutions' (open ONLY if you get stuck!)
#    Tip: Set FORCE_REFRESH_NOTEBOOK = True only if you want to reset your notebooks to a fresh copy.
LAUNCH_CONSOLE = True
FORCE_REFRESH_NOTEBOOK = False
SQL_NOTEBOOK_TITLE = 'Lab 05 — EuroPack SQL Missions'
HINTS_NOTEBOOK_TITLE = 'Lab 05 — 💡 Hints & Solutions'
SQL_MISSION_CELLS = ['-- =========================================================================\n'
 '-- MISSION 0 (START HERE — Your Very First SQL Query! No GROUP BY yet!)\n'
 '-- Business Goal: Before computing any KPIs, inspect our bottling lines and\n'
 '--                preview a few raw breakdown tickets on LINE_B.\n'
 '-- Console Tip: Highlight ANY single query below and press Cmd+Shift+Enter (Mac)\n'
 '--              or Ctrl+Shift+Enter (Windows/Linux) to run ONLY the highlighted part!\n'
 '-- =========================================================================\n'
 '\n'
 '-- 0A. Read specific columns from a table (SELECT ... FROM ...)\n'
 'SELECT line_id, line_name, filler_type, hourly_operating_cost_eur\n'
 'FROM stg_lines;\n'
 '\n'
 '-- 0B. Filter rows (WHERE) and preview just 5 rows (LIMIT)\n'
 'SELECT log_id, operational_shift_date, shift_code, line_code, root_cause, downtime_min\n'
 'FROM stg_stoppages\n'
 "WHERE line_code = 'LINE_B'\n"
 'LIMIT 5;',
 '-- =========================================================================\n'
 '-- MISSION 1 (DEMO — Your First Aggregation with GROUP BY)\n'
 '-- Business Goal: How much total downtime comes from Unplanned breakdowns\n'
 '--                versus scheduled Preventive maintenance?\n'
 '-- =========================================================================\n'
 'SELECT stoppage_type,\n'
 '       COUNT(*)          AS incident_count,\n'
 '       SUM(downtime_min) AS total_downtime_min\n'
 'FROM stg_stoppages\n'
 'GROUP BY stoppage_type\n'
 'ORDER BY total_downtime_min DESC;',
 '-- =========================================================================\n'
 '-- MISSION 2 (WHERE + ORDER BY + LIMIT): Top 5 Longest Unplanned Breakdowns\n'
 '-- Business Goal: Maintenance wants to review the 5 single worst unplanned\n'
 '--                breakdowns first to understand our maximum repair delays.\n'
 '-- Task: Fill in the 3 blanks (___) and run!\n'
 '-- =========================================================================\n'
 'SELECT log_id, line_code, root_cause, downtime_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = '___'\n"
 'ORDER BY ___ DESC, log_id\n'
 'LIMIT ___;',
 '-- =========================================================================\n'
 '-- MISSION 3 (GROUP BY + SUM + AVG): Is LINE_A Really the Worst Machine?\n'
 '-- Business Goal: Management blames LINE_A for most downtime. Rank every\n'
 '--                bottling line by total unplanned downtime minutes (total_min)\n'
 '--                and average repair duration (mttr_min) to test their claim!\n'
 '-- =========================================================================\n'
 'SELECT line_code,\n'
 '       COUNT(*)                    AS stoppages,\n'
 '       SUM(___)                    AS total_min,\n'
 '       ROUND(AVG(downtime_min), 2) AS mttr_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = 'Unplanned'\n"
 'GROUP BY ___\n'
 'ORDER BY ___ DESC;',
 '-- =========================================================================\n'
 '-- MISSION 4A (GROUP BY + HAVING): Isolate Chronic Failure Root Causes\n'
 '-- Business Goal: Maintenance has a tight engineering budget—find ONLY the\n'
 '--                root causes that triggered MORE THAN 20 unplanned breakdowns.\n'
 '-- Reminder: WHERE filters raw rows BEFORE grouping; HAVING filters groups AFTER!\n'
 '-- =========================================================================\n'
 'SELECT root_cause,\n'
 '       ___ AS incident_count,\n'
 '       ___ AS total_min\n'
 'FROM stg_stoppages\n'
 'WHERE ___\n'
 'GROUP BY ___\n'
 'HAVING ___\n'
 'ORDER BY total_min DESC;',
 '-- =========================================================================\n'
 '-- MISSION 4B (Percentage Share %): What % of Downtime Comes from Each Cause?\n'
 '-- Business Goal: The Head of Maintenance wants a Pareto percentage breakdown\n'
 '--                across ALL 4 unplanned root causes so she can justify capital\n'
 '--                investment on the top 2 problems.\n'
 '-- New SQL Pattern: SUM(SUM(downtime_min)) OVER () computes the grand total\n'
 '--                  across all grouped rows so you can divide by it cleanly!\n'
 '-- =========================================================================\n'
 'SELECT root_cause,\n'
 '       COUNT(*)          AS incident_count,\n'
 '       SUM(downtime_min) AS total_min,\n'
 '       ROUND(SUM(downtime_min) * 100.0 / SUM(SUM(___)) OVER (), 2) AS downtime_share_pct\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = '___'\n"
 'GROUP BY root_cause\n'
 'ORDER BY total_min DESC;',
 '-- =========================================================================\n'
 '-- JOIN TYPES PRIMER (Read & Run Demo before Missions 5–6!)\n'
 '--   1. INNER JOIN      -> Keep ONLY rows that match in BOTH tables (A ∩ B).\n'
 '--   2. LEFT JOIN       -> Keep ALL rows from the LEFT table; fill NULLs if\n'
 '--                         no match exists on the right (use COALESCE(..., 0)!).\n'
 '--   3. RIGHT JOIN      -> Mirror of LEFT JOIN (rarely used — put main table on LEFT!).\n'
 '--   4. FULL OUTER JOIN -> Keep ALL rows from BOTH tables (useful for DQ audits).\n'
 '--\n'
 "-- Mini-Demo: Compare INNER vs LEFT JOIN when looking for 'Capper Sensor Drift'\n"
 '--            across our 4 bottling lines! (Highlight either query + Cmd/Ctrl+Shift+Enter!)\n'
 '-- =========================================================================\n'
 '\n'
 '-- INNER JOIN: lines with 0 Capper Sensor Drift incidents DISAPPEAR!\n'
 'SELECT l.line_id, COUNT(s.log_id) AS capper_stops\n'
 'FROM stg_lines l\n'
 'INNER JOIN stg_stoppages s\n'
 "    ON l.line_id = s.line_code AND s.root_cause = 'Capper Sensor Drift'\n"
 'GROUP BY l.line_id\n'
 'ORDER BY l.line_id;\n'
 '\n'
 '-- LEFT JOIN: ALL 4 bottling lines survive (showing 0 when no incident occurred!)\n'
 'SELECT l.line_id, COUNT(s.log_id) AS capper_stops\n'
 'FROM stg_lines l\n'
 'LEFT JOIN stg_stoppages s\n'
 "    ON l.line_id = s.line_code AND s.root_cause = 'Capper Sensor Drift'\n"
 'GROUP BY l.line_id\n'
 'ORDER BY l.line_id;',
 '-- =========================================================================\n'
 '-- MISSION 5 (INNER JOIN): Translate Breakdown Minutes into Euros (€)\n'
 '-- Business Goal: The COO makes investment decisions in Euros, not minutes!\n'
 "--                Enrich stg_stoppages (s) with each line's name, filler type,\n"
 '--                and hourly_operating_cost_eur from stg_lines (l).\n'
 '-- =========================================================================\n'
 'SELECT s.line_code,\n'
 '       l.line_name,\n'
 '       l.filler_type,\n'
 '       COUNT(*)                                       AS stoppages,\n'
 '       SUM(s.downtime_min)                            AS total_min,\n'
 '       ROUND(SUM((s.downtime_min / 60.0) * l.___), 2) AS downtime_cost_eur\n'
 'FROM stg_stoppages s\n'
 'INNER JOIN ___ l\n'
 '    ON s.line_code = l.___\n'
 "WHERE s.stoppage_type = 'Unplanned'\n"
 'GROUP BY s.line_code, l.line_name, l.filler_type\n'
 'ORDER BY downtime_cost_eur DESC;',
 '-- =========================================================================\n'
 '-- MISSION 6A (CTE Building Blocks — Demo, Highlight & Run Each Block!)\n'
 '-- Business Goal: The Plant Director asks:\n'
 '--   "Over 60 days, what % of our scheduled shifts ran COMPLETELY breakdown-free?"\n'
 '--\n'
 '-- To answer this cleanly, we need TWO building blocks before joining:\n'
 '--   Block 1: Every scheduled (date, shift, line) that ran -> 720 total line-shifts.\n'
 '--   Block 2: Total unplanned downtime per (date, shift, line) -> only 210 shifts!\n'
 '--\n'
 '-- Tip: Highlight Block 1 and press Cmd/Ctrl+Shift+Enter, then highlight Block 2 and\n'
 '--      press Cmd/Ctrl+Shift+Enter! In Mission 6B, we wrap both blocks inside a CTE\n'
 '--      (WITH name AS (...)) — think of a CTE as a temporary named worksheet!\n'
 '-- =========================================================================\n'
 '\n'
 '-- Block 1 (all_line_shifts): How many unique line-shifts ran in 60 days? (720 rows!)\n'
 'SELECT DISTINCT operational_shift_date, shift_code, line_id\n'
 'FROM stg_telemetry\n'
 'ORDER BY 1, 2, 3;\n'
 '\n'
 '-- Block 2 (shift_stops): Which line-shifts had an unplanned breakdown? (210 rows!)\n'
 'SELECT operational_shift_date, shift_code, line_code,\n'
 '       SUM(downtime_min) AS unplanned_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = 'Unplanned'\n"
 'GROUP BY operational_shift_date, shift_code, line_code\n'
 'ORDER BY 1, 2, 3;',
 '-- =========================================================================\n'
 '-- MISSION 6B (CTE + LEFT JOIN + COALESCE + % Clean Shifts!)\n'
 '-- Business Goal: Combine Block 1 (720 scheduled shifts) and Block 2 (210 shifts\n'
 '--                with stoppages) using a WITH (CTE) + LEFT JOIN so clean shifts\n'
 '--                are NOT erased, and compute pct_clean_shifts!\n'
 '-- Tip: You can still highlight either inner SELECT inside WITH and press Cmd/Ctrl+Shift+Enter!\n'
 '-- =========================================================================\n'
 'WITH all_line_shifts AS (\n'
 '    SELECT DISTINCT operational_shift_date, shift_code, line_id\n'
 '    FROM stg_telemetry\n'
 '),\n'
 'shift_stops AS (\n'
 '    SELECT operational_shift_date, shift_code, line_code,\n'
 '           SUM(downtime_min) AS unplanned_min\n'
 '    FROM stg_stoppages\n'
 "    WHERE stoppage_type = 'Unplanned'\n"
 '    GROUP BY operational_shift_date, shift_code, line_code\n'
 ')\n'
 'SELECT COUNT(*)                                                          AS total_line_shifts,\n'
 '       COUNT(st.unplanned_min)                                           AS shifts_with_stoppage,\n'
 '       COUNT(*) - COUNT(st.unplanned_min)                                AS clean_shifts,\n'
 '       ROUND((COUNT(*) - COUNT(st.unplanned_min)) * 100.0 / COUNT(*), 2) AS pct_clean_shifts,\n'
 '       SUM(___(st.unplanned_min, 0))                                     AS total_unplanned_min\n'
 'FROM all_line_shifts ls\n'
 '___ JOIN shift_stops st\n'
 '    ON ls.operational_shift_date = st.operational_shift_date\n'
 '   AND ls.shift_code = st.shift_code\n'
 '   AND ls.line_id = st.line_code;',
 '-- =========================================================================\n'
 '-- LIVE CLASSROOM POLL (DEMO — STOP & VOTE BEFORE RUNNING!):\n'
 '-- Business Danger: What happens if someone skips Dimensional Modeling and\n'
 '--                  naively JOINs 15-min stg_telemetry (23,000 rows) directly\n'
 '--                  to stg_stoppages? (Remember: true unplanned downtime = 22,260 min!)\n'
 '-- =========================================================================\n'
 'SELECT COUNT(*)                                 AS joined_rows,\n'
 '       SUM(st.downtime_min)                     AS FAKE_inflated_downtime_min,\n'
 '       ROUND(SUM(st.downtime_min) / 22260.0, 1) AS inflation_multiplier\n'
 'FROM stg_telemetry t\n'
 'JOIN stg_stoppages st\n'
 '  ON t.operational_shift_date = st.operational_shift_date\n'
 ' AND t.shift_code = st.shift_code\n'
 ' AND t.line_id = st.line_code\n'
 "WHERE st.stoppage_type = 'Unplanned';",
 '-- =========================================================================\n'
 '-- GOLD BUILD 1 (DEMO — Run me!): Create Your First Dimension (Dim_Line)\n'
 '-- Business Goal: Turn your repaired schema.dbml into physical Gold Dimension\n'
 '--                tables with clean integer Surrogate Keys (1, 2, 3, 4).\n'
 '-- After running, click Refresh on the left Database Explorer -> see Dim_Line!\n'
 '-- =========================================================================\n'
 'CREATE OR REPLACE TABLE Dim_Line AS\n'
 'SELECT ROW_NUMBER() OVER (ORDER BY line_id) AS line_key,\n'
 '       line_id                              AS line_code,\n'
 '       line_name,\n'
 '       filler_type,\n'
 '       nominal_cycle_time_sec,\n'
 '       hourly_operating_cost_eur\n'
 'FROM stg_lines;\n'
 '\n'
 'SELECT * FROM Dim_Line ORDER BY line_key;',
 '-- =========================================================================\n'
 '-- GOLD BUILD 2 (Fill in the blanks ___): Create Dim_Shift & Dim_Format\n'
 '-- Business Goal: Materialize the Shift (480 min) and Bottle Format Dimensions\n'
 '--                matching your schema.dbml blueprint.\n'
 '-- =========================================================================\n'
 'CREATE OR REPLACE TABLE Dim_Shift AS\n'
 'SELECT ROW_NUMBER() OVER (ORDER BY shift_id) AS shift_key,\n'
 '       shift_id                              AS shift_code,\n'
 '       shift_name,\n'
 '       start_time,\n'
 '       end_time,\n'
 '       (___ * 60)::INTEGER                   AS planned_duration_min\n'
 'FROM stg_shifts;\n'
 '\n'
 'CREATE OR REPLACE TABLE Dim_Format AS\n'
 'SELECT ROW_NUMBER() OVER (ORDER BY ___) AS format_key,\n'
 '       format_code,\n'
 '       format_name,\n'
 '       packaging_material,\n'
 '       target_hourly_output,\n'
 '       unit_reject_cost_eur\n'
 'FROM ___;\n'
 '\n'
 'SELECT * FROM Dim_Shift ORDER BY shift_key;',
 '-- =========================================================================\n'
 '-- GOLD BUILD 3 (Fill in the 2 blanks ___): Create 60-Day Dim_Date\n'
 '-- Business Goal: Build one conformed Calendar Dimension shared across BOTH\n'
 '--                production and breakdowns, with ready-made is_weekend/quarter flags!\n'
 '-- Why integer YYYYMMDD (20261001)? Human-readable, naturally sortable, fast 4-byte\n'
 '-- joins, zero timezone/locale drift, and supports Unknown/TBD sentinel keys without NULL!\n'
 '-- =========================================================================\n'
 'CREATE OR REPLACE TABLE Dim_Date AS\n'
 'WITH bounds AS (\n'
 '    SELECT MIN(operational_shift_date)::DATE AS min_d,\n'
 '           MAX(operational_shift_date)::DATE AS max_d\n'
 '    FROM stg_telemetry\n'
 '),\n'
 'dates AS (\n'
 "    SELECT unnest(generate_series(min_d, max_d, INTERVAL '1 DAY'))::DATE AS cal_date\n"
 '    FROM bounds\n'
 ')\n'
 "SELECT strftime(cal_date, '___')::INTEGER AS date_key,\n"
 '       cal_date                           AS calendar_date,\n'
 '       dayname(cal_date)                  AS day_of_week,\n'
 '       dayofmonth(cal_date)               AS day_of_month,\n'
 '       monthname(cal_date)                AS month_name,\n'
 '       quarter(cal_date)                  AS quarter,\n'
 '       year(cal_date)                     AS year,\n'
 '       CASE WHEN dayofweek(cal_date) IN (0, 6) THEN ___ ELSE FALSE END AS is_weekend\n'
 'FROM dates;\n'
 '\n'
 'SELECT * FROM Dim_Date ORDER BY date_key LIMIT 7;',
 '-- =========================================================================\n'
 '-- SPEED SPRINT #1: Which Packaging Material Suffers the Worst Reject Rate?\n'
 '-- Business Goal: Starting from our 3 Bottle Formats (Dim_Format), compute\n'
 '--                total bottled units, rejected units, and reject_rate_pct!\n'
 '--\n'
 '-- Notice TWO modeling lessons here:\n'
 '--  1. We put FROM Dim_Format FIRST — matching our business question ("Start with\n'
 '--     our formats, then pull their production stats via LEFT JOIN!").\n'
 '--  2. Why join on f.format_code instead of f.format_key today? Because stg_telemetry\n'
 "--     is still a SILVER table (it only knows the ERP code 'GLASS_075'). Next week\n"
 '--     in Lab 06, we run this lookup ONCE to store integer format_key inside\n'
 '--     Fact_Shift_Production — after that, every Gold query joins purely on integers!\n'
 '-- =========================================================================\n'
 'SELECT f.format_code,\n'
 '       f.packaging_material,\n'
 '       SUM(t.bottled_units)                                           AS total_bottled,\n'
 '       SUM(t.rejected_units)                                          AS total_rejected,\n'
 '       ROUND(SUM(t.rejected_units) * 100.0 / SUM(t.bottled_units), 2) AS reject_rate_pct\n'
 'FROM ___ f\n'
 'LEFT JOIN stg_telemetry t\n'
 '    ON f.___ = t.bottle_format\n'
 'GROUP BY f.format_code, f.packaging_material\n'
 'ORDER BY reject_rate_pct DESC;',
 '-- =========================================================================\n'
 '-- SPEED SPRINT #2 & #3: Weekend Output (Starting from Dim_Date!) & Tech Load\n'
 '-- Business Goal (#2): Does bottling volume drop on weekends? Start FROM Dim_Date\n'
 '--                     to compare average daily bottles on Weekends vs Weekdays!\n'
 '-- Business Goal (#3): Which technician_team carries the heaviest unplanned load?\n'
 '-- =========================================================================\n'
 'WITH daily_output AS (\n'
 '    SELECT d.calendar_date,\n'
 '           d.is_weekend,\n'
 '           SUM(t.bottled_units) AS daily_bottles\n'
 '    FROM ___ d\n'
 '    LEFT JOIN stg_telemetry t\n'
 '        ON d.___ = t.operational_shift_date\n'
 '    GROUP BY d.calendar_date, d.is_weekend\n'
 ')\n'
 'SELECT is_weekend,\n'
 '       COUNT(*)                     AS day_count,\n'
 '       ROUND(AVG(daily_bottles), 0) AS avg_daily_bottles\n'
 'FROM daily_output\n'
 'GROUP BY is_weekend\n'
 'ORDER BY is_weekend;\n'
 '\n'
 '-- Sprint #3 (Highlight & press Cmd/Ctrl+Shift+Enter to run!):\n'
 'SELECT technician_team,\n'
 '       COUNT(*)          AS stoppages,\n'
 '       SUM(downtime_min) AS total_min\n'
 'FROM stg_stoppages\n'
 "WHERE stoppage_type = '___'\n"
 'GROUP BY ___\n'
 'ORDER BY total_min DESC;']
SQL_HINT_CELLS = ['-- 💡 MISSION 2 (HINT & SOLUTION) — Top 5 Longest Unplanned Breakdowns\n'
 '-- TARGET: All 5 rows should show downtime_min = 180 (3 full hours!).\n'
 "-- HINT: Filter stoppage_type = 'Unplanned', sort by downtime_min DESC, LIMIT 5.\n"
 '--\n'
 '-- SOLUTION:\n'
 '-- SELECT log_id, line_code, root_cause, downtime_min\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- ORDER BY downtime_min DESC, log_id\n'
 '-- LIMIT 5;',
 '-- 💡 MISSION 3 (HINT & SOLUTION) — Unplanned Downtime & MTTR by Line\n'
 "-- TARGET: Surprise! LINE_B is #1 with 81 stops and 8,385 min (nearly 2x LINE_A's 4,755 min!).\n"
 '-- HINT: SUM(downtime_min) | GROUP BY line_code | ORDER BY total_min DESC.\n'
 '--\n'
 '-- SOLUTION:\n'
 '-- SELECT line_code,\n'
 '--        COUNT(*)                    AS stoppages,\n'
 '--        SUM(downtime_min)           AS total_min,\n'
 '--        ROUND(AVG(downtime_min), 2) AS mttr_min\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY line_code\n'
 '-- ORDER BY total_min DESC;',
 '-- 💡 MISSION 4A (HINT & SOLUTION) — Chronic Root Causes (> 20 Stops)\n'
 '-- TARGET: Exactly 2 causes survive: Filler Valve Seal Leak (106 stops, 12,030 min)\n'
 '--         and Pasteurizer Thermal Trip (91 stops, 9,660 min).\n'
 '-- HINT: COUNT(*) AS incident_count | SUM(downtime_min) AS total_min |\n'
 "--       WHERE stoppage_type = 'Unplanned' | GROUP BY root_cause | HAVING COUNT(*) > 20.\n"
 '--\n'
 '-- SOLUTION:\n'
 '-- SELECT root_cause,\n'
 '--        COUNT(*)          AS incident_count,\n'
 '--        SUM(downtime_min) AS total_min\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY root_cause\n'
 '-- HAVING COUNT(*) > 20\n'
 '-- ORDER BY total_min DESC;',
 '-- 💡 MISSION 4B (HINT & SOLUTION) — Percentage Share of Unplanned Downtime\n'
 '-- TARGET: Filler Valve Seal Leak = 54.04% | Pasteurizer Thermal Trip = 43.40%\n'
 '--         (Together = 97.44% of all unplanned downtime! Remaining 2 causes = 1.35% & 1.21%.)\n'
 "-- HINT: Fill SUM(SUM(downtime_min)) OVER () and stoppage_type = 'Unplanned'.\n"
 '--\n'
 '-- SOLUTION:\n'
 '-- SELECT root_cause,\n'
 '--        COUNT(*)          AS incident_count,\n'
 '--        SUM(downtime_min) AS total_min,\n'
 '--        ROUND(SUM(downtime_min) * 100.0 / SUM(SUM(downtime_min)) OVER (), 2) AS downtime_share_pct\n'
 '-- FROM stg_stoppages\n'
 "-- WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY root_cause\n'
 '-- ORDER BY total_min DESC;',
 '-- 💡 MISSION 5 (HINT & SOLUTION) — INNER JOIN Downtime Cost in Euros\n'
 "-- TARGET: LINE_B is #1 at €29,347.50 (€10k more than LINE_A's €19,020.00!).\n"
 '-- HINT: l.hourly_operating_cost_eur | INNER JOIN stg_lines l ON s.line_code = l.line_id.\n'
 '--\n'
 '-- SOLUTION:\n'
 '-- SELECT s.line_code,\n'
 '--        l.line_name,\n'
 '--        l.filler_type,\n'
 '--        COUNT(*)                                                             AS stoppages,\n'
 '--        SUM(s.downtime_min)                                                  AS total_min,\n'
 '--        ROUND(SUM((s.downtime_min / 60.0) * l.hourly_operating_cost_eur), 2) AS downtime_cost_eur\n'
 '-- FROM stg_stoppages s\n'
 '-- INNER JOIN stg_lines l\n'
 '--     ON s.line_code = l.line_id\n'
 "-- WHERE s.stoppage_type = 'Unplanned'\n"
 '-- GROUP BY s.line_code, l.line_name, l.filler_type\n'
 '-- ORDER BY downtime_cost_eur DESC;',
 '-- 💡 MISSION 6B (HINT & SOLUTION) — CTE + LEFT JOIN + % Clean Shifts\n'
 '-- TARGET: total_line_shifts = 720 | shifts_with_stoppage = 210 | clean_shifts = 510 |\n'
 '--         pct_clean_shifts = 70.83% | total_unplanned_min = 22,260.\n'
 '--         (Try temporarily changing LEFT -> INNER to see 510 clean shifts vanish and pct drop to 0%!)\n'
 '-- HINT: Fill COALESCE(st.unplanned_min, 0) and LEFT JOIN shift_stops st.\n'
 '--\n'
 '-- SOLUTION:\n'
 '-- WITH all_line_shifts AS (\n'
 '--     SELECT DISTINCT operational_shift_date, shift_code, line_id\n'
 '--     FROM stg_telemetry\n'
 '-- ),\n'
 '-- shift_stops AS (\n'
 '--     SELECT operational_shift_date, shift_code, line_code,\n'
 '--            SUM(downtime_min) AS unplanned_min\n'
 '--     FROM stg_stoppages\n'
 "--     WHERE stoppage_type = 'Unplanned'\n"
 '--     GROUP BY operational_shift_date, shift_code, line_code\n'
 '-- )\n'
 '-- SELECT COUNT(*)                                                          AS total_line_shifts,\n'
 '--        COUNT(st.unplanned_min)                                           AS shifts_with_stoppage,\n'
 '--        COUNT(*) - COUNT(st.unplanned_min)                                AS clean_shifts,\n'
 '--        ROUND((COUNT(*) - COUNT(st.unplanned_min)) * 100.0 / COUNT(*), 2) AS pct_clean_shifts,\n'
 '--        SUM(COALESCE(st.unplanned_min, 0))                                AS total_unplanned_min\n'
 '-- FROM all_line_shifts ls\n'
 '-- LEFT JOIN shift_stops st\n'
 '--     ON ls.operational_shift_date = st.operational_shift_date\n'
 '--    AND ls.shift_code = st.shift_code\n'
 '--    AND ls.line_id = st.line_code;',
 '-- 💡 GOLD BUILD 2 (HINT & SOLUTION) — Create Dim_Shift & Dim_Format\n'
 '-- TARGET: Dim_Shift = 3 rows (planned_duration_min = 480); Dim_Format = 3 rows.\n'
 '-- HINT: (planned_duration_hours * 60)::INTEGER | ORDER BY format_code | FROM stg_formats.\n'
 '--\n'
 '-- SOLUTION:\n'
 '-- CREATE OR REPLACE TABLE Dim_Shift AS\n'
 '-- SELECT ROW_NUMBER() OVER (ORDER BY shift_id) AS shift_key,\n'
 '--        shift_id                              AS shift_code,\n'
 '--        shift_name, start_time, end_time,\n'
 '--        (planned_duration_hours * 60)::INTEGER AS planned_duration_min\n'
 '-- FROM stg_shifts;\n'
 '--\n'
 '-- CREATE OR REPLACE TABLE Dim_Format AS\n'
 '-- SELECT ROW_NUMBER() OVER (ORDER BY format_code) AS format_key,\n'
 '--        format_code, format_name, packaging_material,\n'
 '--        target_hourly_output, unit_reject_cost_eur\n'
 '-- FROM stg_formats;',
 '-- 💡 GOLD BUILD 3 (HINT & SOLUTION) — Create 60-Day Dim_Date\n'
 '-- TARGET: 60 rows (2026-10-01 .. 2026-11-29), date_key = 20261001..20261129, 18 weekend days.\n'
 "-- HINT: strftime(cal_date, '%Y%m%d')::INTEGER and THEN TRUE ELSE FALSE.\n"
 '--\n'
 '-- SOLUTION:\n'
 '-- CREATE OR REPLACE TABLE Dim_Date AS\n'
 '-- WITH bounds AS (\n'
 '--     SELECT MIN(operational_shift_date)::DATE AS min_d,\n'
 '--            MAX(operational_shift_date)::DATE AS max_d\n'
 '--     FROM stg_telemetry\n'
 '-- ),\n'
 '-- dates AS (\n'
 "--     SELECT unnest(generate_series(min_d, max_d, INTERVAL '1 DAY'))::DATE AS cal_date\n"
 '--     FROM bounds\n'
 '-- )\n'
 "-- SELECT strftime(cal_date, '%Y%m%d')::INTEGER AS date_key,\n"
 '--        cal_date AS calendar_date, dayname(cal_date) AS day_of_week,\n'
 '--        dayofmonth(cal_date) AS day_of_month, monthname(cal_date) AS month_name,\n'
 '--        quarter(cal_date) AS quarter, year(cal_date) AS year,\n'
 '--        CASE WHEN dayofweek(cal_date) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend\n'
 '-- FROM dates;',
 '-- 💡 SPEED SPRINT #1 (HINT & SOLUTION) — Reject Rate by Bottle Format\n'
 '-- TARGET: GLASS_075 (Amber Glass 450g) = 4.29% reject rate (more than 2x PET_05 1.84% / PET_15 1.87%!).\n'
 '-- HINT: FROM Dim_Format f LEFT JOIN stg_telemetry t ON f.format_code = t.bottle_format.\n'
 '--\n'
 '-- SOLUTION:\n'
 '-- SELECT f.format_code,\n'
 '--        f.packaging_material,\n'
 '--        SUM(t.bottled_units)                                           AS total_bottled,\n'
 '--        SUM(t.rejected_units)                                          AS total_rejected,\n'
 '--        ROUND(SUM(t.rejected_units) * 100.0 / SUM(t.bottled_units), 2) AS reject_rate_pct\n'
 '-- FROM Dim_Format f\n'
 '-- LEFT JOIN stg_telemetry t\n'
 '--     ON f.format_code = t.bottle_format\n'
 '-- GROUP BY f.format_code, f.packaging_material\n'
 '-- ORDER BY reject_rate_pct DESC;',
 '-- 💡 SPEED SPRINT #2 & #3 (HINT & SOLUTION) — Weekend Output & Tech Load\n'
 '-- TARGET #2: Weekdays (42 days) = ~338,449 bottles/day vs Weekends (18 days) = ~340,060/day (truly 24/7!).\n'
 '-- TARGET #3: Maint_Mechanical absorbs 21,690 min (97.4%) vs Maint_Controls 570 min!\n'
 '-- HINT: FROM Dim_Date d LEFT JOIN stg_telemetry t ON d.calendar_date = t.operational_shift_date;\n'
 "--       Sprint #3: WHERE stoppage_type = 'Unplanned' GROUP BY technician_team.\n"
 '--\n'
 '-- SOLUTION (#2):\n'
 '-- WITH daily_output AS (\n'
 '--     SELECT d.calendar_date, d.is_weekend, SUM(t.bottled_units) AS daily_bottles\n'
 '--     FROM Dim_Date d\n'
 '--     LEFT JOIN stg_telemetry t ON d.calendar_date = t.operational_shift_date\n'
 '--     GROUP BY d.calendar_date, d.is_weekend\n'
 '-- )\n'
 '-- SELECT is_weekend, COUNT(*) AS day_count, ROUND(AVG(daily_bottles), 0) AS avg_daily_bottles\n'
 '-- FROM daily_output GROUP BY is_weekend ORDER BY is_weekend;\n'
 '--\n'
 '-- SOLUTION (#3):\n'
 '-- SELECT technician_team, COUNT(*) AS stoppages, SUM(downtime_min) AS total_min\n'
 "-- FROM stg_stoppages WHERE stoppage_type = 'Unplanned'\n"
 '-- GROUP BY technician_team ORDER BY total_min DESC;']

if LAUNCH_CONSOLE:
    try:
        con.sql("CALL start_ui_server();")
    except Exception:
        pass
    # Seed Hints & Solutions in the sidebar first (inactive), then activate the clean Missions notebook!
    seeded_h = seed_duckdb_ui_notebook(HINTS_NOTEBOOK_TITLE, SQL_HINT_CELLS, force_refresh=FORCE_REFRESH_NOTEBOOK, make_active=False)
    seeded_m = seed_duckdb_ui_notebook(SQL_NOTEBOOK_TITLE, SQL_MISSION_CELLS, force_refresh=FORCE_REFRESH_NOTEBOOK, make_active=True)
    con.sql("CALL start_ui();")
    if not seeded_m:
        seed_duckdb_ui_notebook(HINTS_NOTEBOOK_TITLE, SQL_HINT_CELLS, force_refresh=FORCE_REFRESH_NOTEBOOK, make_active=False)
        seed_duckdb_ui_notebook(SQL_NOTEBOOK_TITLE, SQL_MISSION_CELLS, force_refresh=FORCE_REFRESH_NOTEBOOK, make_active=True)
    print("\n🚀 DuckDB SQL Console is LIVE at http://localhost:4213")
    print(f"   • Main workbook (active): '{SQL_NOTEBOOK_TITLE}' ({len(SQL_MISSION_CELLS)} clean cells)")
    print(f"   • Stuck? Open in sidebar: '{HINTS_NOTEBOOK_TITLE}' ({len(SQL_HINT_CELLS)} labeled hint/solution cells)")
    print("   • Shortcut: Highlight any single SQL block + press Cmd+Shift+Enter (Mac) / Ctrl+Shift+Enter (Windows/Linux) to run just that block!")
    print("   • Offline backup SQL file:", "lab_05_sql_missions_en.sql")
else:
    print("ℹ️ Set LAUNCH_CONSOLE = True and run this cell to launch DuckDB SQL Console (http://localhost:4213)!")


## 3. SQL Missions & Dimension Build *(Work Inside DuckDB Console!)*
Switch to your **DuckDB SQL Console** (`http://localhost:4213`) and work top-to-bottom through **`Lab 05 — EuroPack SQL Missions`** (backup file: `lab_05_sql_missions_en.sql`). If you ever get stuck on a mission, open the separate **`Lab 05 — 💡 Hints & Solutions`** notebook in the left sidebar—each cell is clearly labeled by mission name!

> **Console Pro Tip:** Highlight *any* single SQL block (even one inner `SELECT` inside a `WITH` CTE!) and press **`Cmd + Shift + Enter` (Mac)** or **`Ctrl + Shift + Enter` (Windows/Linux)** to run just the highlighted part!

| Mission | Skill Practiced | Key Business Finding |
| :--- | :--- | :--- |
| **Mission 0** *(Demo)* | Basic `SELECT`, `FROM`, `WHERE`, `LIMIT` | Read machine catalog & preview `LINE_B` stoppages |
| **Mission 1** *(Demo)* | First `GROUP BY` (`COUNT`, `SUM`) | `Unplanned`: `211` / `22,260 min` · `Preventive`: `78` / `3,510 min` |
| **Mission 2** | `WHERE`, `ORDER BY`, `LIMIT` | Top 5 longest unplanned breakdowns all lasted **`180 min`** |
| **Mission 3** | `GROUP BY` + `AVG` (`MTTR`) | **`LINE_B`** is #1 (`81` stops, **`8,385 min`**) — nearly 2× `LINE_A`! |
| **Mission 4A** | `GROUP BY` + `HAVING` | Only **2 chronic causes** `> 20` stops (`106` & `91` stops) |
| **Mission 4B** | Percentage Share (`OVER ()`) | Top 2 causes = **`54.04% + 43.40% = 97.44%`** of unplanned downtime! |
| **JOIN Primer + M5** | 4 JOIN Types + `INNER JOIN` (Euros!) | **`LINE_B`** burned **`€29,347.50`** (`€10k` more than `LINE_A`!) |
| **Mission 6A → 6B** | CTE Building Blocks + `LEFT JOIN` + `COALESCE` | **`510 / 720`** clean shifts (**`70.83%`** breakdown-free!) |
| **🗳️ Live Poll** | Fan-Out Explosion *(Vote first!)* | Naive telemetry join inflates downtime **`31.9×`** to **`710,400 min`**! |
| **Gold Builds 1–3** | `CREATE OR REPLACE TABLE Dim_*` | Build **`Dim_Line` (`4`)**, **`Dim_Shift` (`3`)**, **`Dim_Format` (`3`)**, **`Dim_Date` (`60`)** |
| **⚡ Sprints #1–#3** | `FROM Dim_Format` & `FROM Dim_Date` (`LEFT JOIN`) | `GLASS_075` = **`4.29%`** reject · Weekends ≈ Weekdays · `Maint_Mechanical` = `97.4%` |

In [ ]:
# Complete Missions 0–6B, the Live Fan-Out Poll, Gold Builds 1–3, and the Speed Sprint
# directly inside the DuckDB SQL Console (http://localhost:4213)!
#
# Run this cell anytime to see which tables currently exist in europack_lab05.duckdb:
print("Current tables in europack_lab05.duckdb:")
print(con.execute("SHOW TABLES").df().to_string(index=False))


## 4. Final Checkpoint: Verify Your 4 Dimensions, Export Parquet & Close
Once you have created `Dim_Line`, `Dim_Shift`, `Dim_Format`, and `Dim_Date` in the DuckDB SQL Console (Gold Builds 1–3), run this cell to validate all row counts, export `dim_*.parquet`, stop the Console cleanly, and close `europack_lab05.duckdb` for **Lab 06**!

In [ ]:
# Run this final cell once you have built Dim_Line, Dim_Shift, Dim_Format, and Dim_Date in DuckDB Console!
ok = verify_and_export_dimensions()

# Stop the DuckDB SQL Console cleanly (if it was started) and release the file lock for Lab 06
try:
    con.sql("CALL stop_ui_server();")
except Exception:
    pass

tables = con.execute("SELECT table_name FROM information_schema.tables WHERE table_type='BASE TABLE' ORDER BY 1").df()["table_name"].tolist()
con.close()
print("\nClosed:", DB_PATH)
print("Tables saved for Lab 06:", tables)
